# Day 2 lab: How Spark plans a query: Catalyst and Tungsten

**Goal:** Name every decision between your code and the running tasks: which rewrites Catalyst makes, how physical operators are chosen, and how Tungsten executes them.

**Time:** about 2 h 30 min. Checklist: 1.6, 1.8, 1.13 · 🧭 planning decisions.

Put [`lab_utils.py`](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py) next to this notebook, run the setup cell, then work through the tasks in order. Write your code in the empty cell under each task and your observations in the notes cell. Keep the Spark UI open (the setup cell prints its address).

In [ ]:
from lab_utils import *

spark = get_spark()

## 📘 Code along

Type the code yourself instead of pasting it, run every cell, and keep the Spark UI open. Swap in `spark = get_spark()` from the starter kit wherever an example builds its own session.

### Task 1. Write the lab dataset once — 10 min

`make_orders(spark, 5_000_000).write.parquet("data/orders")` and `make_customers(spark).write.parquet("data/customers")`. Later days reuse both folders. Reading from Parquet shows pushdown and pruning, which generated data cannot.

In [ ]:
# Task 1: your code here


*Task 1 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 2. Chapter 6: Old SQL, New Tricks — 30 min

Type every pair of examples in the official PySpark guide: SELECT and FILTER, JOIN, GROUP BY, a window with RANK, UNION, SET configurations, listing tables, `selectExpr`, temp views, and a Python UDF registered for SQL. After each SQL / DataFrame pair, call `explain()` on both and confirm the physical plans match: one engine, two syntaxes. The chapter creates Parquet tables named `people` and `orders` in `spark-warehouse/`; drop them at the end (`DROP TABLE people`, `DROP TABLE orders`) so they do not get mixed up with your own data.

[Chapter 6: Old SQL, New Tricks](https://spark.apache.org/docs/latest/api/python/user_guide/sql.html) · [Notebook source on GitHub](https://github.com/apache/spark/blob/v4.2.0/python/docs/source/user_guide/sql.ipynb)

In [ ]:
# Task 2: your code here


*Task 2 notes: what you predicted, what the plan and the Spark UI showed.*

## ✍️ Exercises

Write these from scratch in the cells below, using the patterns you just typed. Each one ends with a Check you can verify yourself; open the reference solution only after you have a working answer.

### Task 3. Exercise 1: one question in SQL and in the DataFrame API — 30 min

Question: revenue of paid orders per country, top 5. Register `data/orders` and `data/customers` as temp views and answer it with `spark.sql`, then again with the DataFrame API (`filter`, `join`, `groupBy`, `agg`, `orderBy`, `limit`). 

**Check:** `assertDataFrameEqual(sql_df, api_df, checkRowOrder=True)` passes, and the operator names in both `explain("formatted")` trees are identical and in the same order (only the `#ids` differ). Proves 1.6, 1.13.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day02.py)

In [ ]:
# Task 3: your code here


*Task 3 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 4. Exercise 2: port the official wordcount.py from RDDs to DataFrames — 30 min

Download a text file (`curl -o README.md https://raw.githubusercontent.com/apache/spark/v4.2.0/README.md`), run the logic of the official `wordcount.py` in your notebook (`flatMap`, `map`, `reduceByKey`), then rewrite it with `spark.read.text`, `F.explode(F.split("value", " "))` and `groupBy("word").count()`. 

**Check:** both give the same count for the 10 most frequent words, and the DataFrame plan has `partial_count` before its Exchange: the same map-side combine that `reduceByKey` does. In the Stages tab, compare task times: the RDD version ships every line to Python workers. Proves 1.13, 1.5.

[wordcount.py on GitHub](https://github.com/apache/spark/blob/v4.2.0/examples/src/main/python/wordcount.py) · [Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day02.py)

In [ ]:
# Task 4: your code here


*Task 4 notes: what you predicted, what the plan and the Spark UI showed.*

## 🔬 Prove it

Short, focused checks of today's checklist questions in the plan and the Spark UI.

### Task 5. Four plans, and the rules that made them — 35 min

Query: read `data/orders`, filter `F.col("amount") > 50`, join `data/customers` on customer_id, `groupBy("country").agg(F.sum("amount"))`. Run `explain("extended")` and label the sections: Parsed (names still unresolved), Analyzed (types and ids such as `amount#3`), Optimized (filter moved below the join, `isnotnull(customer_id)` added) and Physical (FileScan with `PushedFilters`, `BroadcastHashJoin`, partial and final `HashAggregate`, one `Exchange`). Then `spark.conf.set("spark.sql.planChangeLog.level", "WARN")`, rebuild a deliberately wasteful version (`select("*")` then two columns, the filter after the join, an extra `F.lit(1) + F.lit(1) == 2`) and `explain()` it: the console lists every rule that changed the plan, such as `ResolveReferences`, `PushDownPredicates`, `ColumnPruning`, `ConstantFolding`, `InferFiltersFromConstraints` and `EnsureRequirements`. Set the level back to `TRACE` afterwards. Proves 1.6.

In [ ]:
# Task 5: your code here


*Task 5 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 6. Blind the optimizer with a UDF — 15 min

Replace `F.col("amount") > 50` with a Python UDF, `udf(lambda a: a is not None and a > 50, "boolean")("amount")`, and compare the scan node: the predicate disappears from `PushedFilters` and an `ArrowEvalPython` node appears (Spark 4.2 runs plain Python UDFs through Arrow by default; `useArrow=False` shows the older `BatchEvalPython`). Time both versions with `run()`. Proves 1.6; preview of 6.1.

In [ ]:
# Task 6: your code here


*Task 6 notes: what you predicted, what the plan and the Spark UI showed.*